# 드라마 장르별 롱폼 영상 통계 수집
YouTube Data API v3로 장르·키워드별 롱폼(10분 이상) 영상 데이터를 수집해 MySQL에 저장합니다.

## 0. 환경 설정

In [1]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from sqlalchemy import create_engine, text
import pymysql

load_dotenv()

api_key    = os.getenv('YOUTUBE_API_KEY')
mysql_host = os.getenv('MYSQL_HOST', 'localhost')
mysql_user = os.getenv('MYSQL_USER', 'root')
mysql_pw   = os.getenv('MYSQL_PASSWORD', '')
mysql_db   = os.getenv('MYSQL_DB', 'youtube_data')

assert api_key, '.env에 YOUTUBE_API_KEY가 없습니다.'

print(f'API KEY : {api_key[:8]}...')
print(f'MySQL   : {mysql_user}@{mysql_host}/{mysql_db}')

API KEY : AIzaSyCu...
MySQL   : root@localhost/youtube_data


## 1. YouTube API 클라이언트 초기화

In [2]:
def init_youtube_client(api_key: str):
    """YouTube Data API v3 클라이언트를 생성한다."""
    try:
        youtube_client = build('youtube', 'v3', developerKey=api_key)
        print('YouTube 클라이언트 초기화 완료')
        return youtube_client
    except Exception as e:
        print(f'[ERROR] 클라이언트 초기화 실패: {e}')
        raise


youtube_client = init_youtube_client(api_key)

YouTube 클라이언트 초기화 완료


## 2. MySQL 연결

In [3]:
def init_db_engine(host: str, user: str, pw: str, db: str):
    """MySQL DB가 없으면 생성하고 SQLAlchemy 엔진을 반환한다."""
    try:
        # DB 없으면 먼저 생성
        conn = pymysql.connect(host=host, user=user, password=pw, charset='utf8mb4')
        with conn.cursor() as cur:
            cur.execute(
                f'CREATE DATABASE IF NOT EXISTS `{db}` '
                f'CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci'
            )
        conn.close()

        url    = f'mysql+pymysql://{user}:{pw}@{host}/{db}?charset=utf8mb4'
        engine = create_engine(url)

        with engine.connect() as c:
            c.execute(text('SELECT 1'))

        print(f'MySQL 연결 성공: {user}@{host}/{db}')
        return engine
    except Exception as e:
        print(f'[ERROR] MySQL 연결 실패: {e}')
        raise


engine = init_db_engine(mysql_host, mysql_user, mysql_pw, mysql_db)

MySQL 연결 성공: root@localhost/youtube_data


## 3. 검색 키워드 정의

In [4]:
# 장르별 검색 키워드 매핑
GENRE_KEYWORDS = {
    '로맨스':   ['눈물의 여왕', '킹더랜드'],
    '재벌':     ['재벌집 막내아들', '상속자들'],
    '불륜복수': ['부부의 세계', '펜트하우스'],
}

LONGFORM_MIN_SECONDS = 600   # 롱폼 기준: 10분 이상
SEARCH_MAX_RESULTS   = 50    # 키워드당 최대 검색 수

print('검색 키워드 목록:')
for genre, keywords in GENRE_KEYWORDS.items():
    print(f'  [{genre}] {" | ".join(keywords)}')

검색 키워드 목록:
  [로맨스] 눈물의 여왕 | 킹더랜드
  [재벌] 재벌집 막내아들 | 상속자들
  [불륜복수] 부부의 세계 | 펜트하우스


## 4. 키워드별 영상 ID 검색

In [5]:
def search_videos_by_keyword(client, keyword: str, max_results: int = 50) -> list[str]:
    """키워드로 YouTube 동영상을 검색하고 video_id 목록을 반환한다."""
    video_ids = []
    try:
        search_results = client.search().list(
            q                 = keyword,
            type              = 'video',
            part              = 'id',
            maxResults        = max_results,
            order             = 'relevance',
            relevanceLanguage = 'ko',
            regionCode        = 'KR'
        ).execute()

        for item in search_results.get('items', []):
            video_ids.append(item['id']['videoId'])

    except Exception as e:
        print(f'[ERROR] 검색 실패 ({keyword}): {e}')

    return video_ids

## 5. 영상 상세 정보 수집

In [6]:
def parse_duration_to_seconds(iso_duration: str) -> int:
    """ISO 8601 재생시간 문자열(PT1H30M45S)을 초 단위 정수로 변환한다."""
    if not iso_duration:
        return 0
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', iso_duration)
    if not match:
        return 0
    h = int(match.group(1) or 0)
    m = int(match.group(2) or 0)
    s = int(match.group(3) or 0)
    return h * 3600 + m * 60 + s


def fetch_video_details(client, keyword: str, genre: str, video_ids: list[str]) -> list[dict]:
    """video_id 목록으로 영상 상세 정보(통계, 재생시간)를 50개씩 배치 수집한다."""
    video_list = []

    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i + 50]
        try:
            res = client.videos().list(
                id   = ','.join(chunk),
                part = 'snippet,statistics,contentDetails'
            ).execute()

            for item in res.get('items', []):
                snippet  = item.get('snippet', {})
                stats    = item.get('statistics', {})
                content  = item.get('contentDetails', {})

                duration_sec = parse_duration_to_seconds(content.get('duration', ''))
                video_id     = item['id']

                video_list.append({
                    'keyword':       keyword,
                    'genre':         genre,
                    'video_id':      video_id,
                    'title':         snippet.get('title', ''),
                    'channel':       snippet.get('channelTitle', ''),
                    'published_at':  snippet.get('publishedAt', ''),
                    'duration_sec':  duration_sec,
                    'view_count':    stats.get('viewCount'),
                    'like_count':    stats.get('likeCount'),
                    'comment_count': stats.get('commentCount'),
                    'url':           f'https://www.youtube.com/watch?v={video_id}'
                })

        except Exception as e:
            print(f'[ERROR] 상세 수집 실패 (chunk {i}~{i+50}): {e}')

    return video_list

## 6. 장르별 전체 수집 실행

In [7]:
# 장르 → 키워드 순서로 순회하며 영상 수집
all_video_list = []

for genre, keywords in GENRE_KEYWORDS.items():
    for keyword in keywords:
        print(f'[{genre}] "{keyword}" 검색 중...')

        video_ids    = search_videos_by_keyword(youtube_client, keyword, SEARCH_MAX_RESULTS)
        search_results = fetch_video_details(youtube_client, keyword, genre, video_ids)

        all_video_list.extend(search_results)
        print(f'       수집: {len(search_results)}개')

print(f'\n전체 수집 합계: {len(all_video_list)}개')

[로맨스] "눈물의 여왕" 검색 중...
       수집: 50개
[로맨스] "킹더랜드" 검색 중...
       수집: 50개
[재벌] "재벌집 막내아들" 검색 중...
       수집: 50개
[재벌] "상속자들" 검색 중...
       수집: 50개
[불륜복수] "부부의 세계" 검색 중...
       수집: 50개
[불륜복수] "펜트하우스" 검색 중...
       수집: 50개

전체 수집 합계: 300개


## 7. 롱폼 필터링 및 DataFrame 정리

In [8]:
def build_longform_dataframe(video_list: list[dict], min_seconds: int = 600) -> pd.DataFrame:
    """10분 이상 영상만 필터링하고 컬럼 타입을 정리한 DataFrame을 반환한다."""
    df = pd.DataFrame(video_list)

    # 롱폼 필터링 (기준 초 이상)
    df = df[df['duration_sec'] >= min_seconds].copy()

    # 수치형: 결측치 0으로 채우고 정수 변환
    num_cols = ['view_count', 'like_count', 'comment_count']
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # 문자형: 결측치 빈 문자열로 채우기
    str_cols = ['keyword', 'genre', 'video_id', 'title', 'channel', 'published_at', 'url']
    for col in str_cols:
        df[col] = df[col].fillna('')

    # 최종 컬럼 순서 정렬 및 duration_sec 제거
    final_cols = ['keyword', 'genre', 'video_id', 'title', 'channel',
                  'published_at', 'view_count', 'like_count', 'comment_count', 'url']
    df = df[final_cols]

    # 중복 video_id 제거 (동일 영상이 여러 키워드에서 검색된 경우 첫 번째 유지)
    df = df.drop_duplicates(subset='video_id').reset_index(drop=True)

    return df


df = build_longform_dataframe(all_video_list, min_seconds=LONGFORM_MIN_SECONDS)

print(f'롱폼 필터링 후: {len(df)}개')
print('\n장르별 영상 수:')
print(df['genre'].value_counts())
df.head()

롱폼 필터링 후: 137개

장르별 영상 수:
genre
재벌      67
불륜복수    53
로맨스     17
Name: count, dtype: int64


,keyword,genre,video_id,title,channel,published_at,view_count,like_count,comment_count,url
0,눈물의 여왕,로맨스,TMQCqU2tZa8,🔥최고 시청률 28.4%!!🔥재산 1조 재벌 3세 '김지원'과 과수원집 아들 '김수...,무비N,2025-07-29T10:00:46Z,1096231,8825,275,https://www.youtube.com/watch?v=TMQCqU2tZa8
1,눈물의 여왕,로맨스,4KqZ55jnqSc,[#눈물의여왕] 1살 오빠 김수현 앞에서만 말랑해지는 앙큼폭스 연하 김지원 ㅋㅋ 보...,샾잉 #ing,2024-04-03T11:01:00Z,1193928,8774,218,https://www.youtube.com/watch?v=4KqZ55jnqSc
2,킹더랜드,로맨스,vkz0m_W1UuY,이걸 아직도 안봤어!? 시청률만 무려 13.8%!! 윤아X이준호 주연 개꿀잼 드라마...,유튜브드라마 : YOUTUBE DRAMA,2024-08-18T06:02:15Z,2491254,13616,378,https://www.youtube.com/watch?v=vkz0m_W1UuY
3,킹더랜드,로맨스,2BHnmGaoY64,공개되자마자 넷플릭스 글로벌 1위를 차지한 이준호X임윤아 주연의 명작 한국 드라마 ...,리뷰 한사발,2026-05-03T01:00:05Z,345708,1549,45,https://www.youtube.com/watch?v=2BHnmGaoY64
4,킹더랜드,로맨스,MauaR1JoHAQ,(1시간) ＂무개념이 뭔지 잘 봤습니다?＂ 직원들 괴롭히는 갑질 상사 혼쭐내주는 본...,DRAMA Voyage,2026-04-16T03:16:33Z,468305,2378,71,https://www.youtube.com/watch?v=MauaR1JoHAQ


## 8. MySQL 저장

In [9]:
def save_to_mysql(df: pd.DataFrame, engine, table: str = 'longform_dramas') -> None:
    """DataFrame을 MySQL 테이블에 저장한다. 기존 테이블은 대체한다."""
    try:
        df.to_sql(
            name      = table,
            con       = engine,
            if_exists = 'replace',
            index     = False,
            chunksize = 500
        )
        with engine.connect() as conn:
            count = conn.execute(text(f'SELECT COUNT(*) FROM `{table}`')).scalar()
        print(f'저장 완료: {count}개 레코드 → {mysql_db}.{table}')
    except Exception as e:
        print(f'[ERROR] MySQL 저장 실패: {e}')
        raise


save_to_mysql(df, engine)

저장 완료: 137개 레코드 → youtube_data.longform_dramas


## 9. 저장 결과 확인

In [10]:
def load_genre_summary(engine, table: str = 'longform_dramas') -> pd.DataFrame:
    """장르·키워드별 영상 수와 평균 조회수를 조회한다."""
    try:
        summary = pd.read_sql(
            text(f'''
                SELECT
                    genre,
                    keyword,
                    COUNT(*)               AS video_count,
                    ROUND(AVG(view_count))    AS avg_views,
                    ROUND(AVG(like_count))    AS avg_likes,
                    ROUND(AVG(comment_count)) AS avg_comments
                FROM `{table}`
                GROUP BY genre, keyword
                ORDER BY genre, keyword
            '''),
            con=engine
        )
        return summary
    except Exception as e:
        print(f'[ERROR] 요약 조회 실패: {e}')
        raise


summary = load_genre_summary(engine)
print('=== 장르·키워드별 롱폼 영상 요약 ===')
display(summary)

=== 장르·키워드별 롱폼 영상 요약 ===


,genre,keyword,video_count,avg_views,avg_likes,avg_comments
0,로맨스,눈물의 여왕,2,1145080.0,8800.0,247.0
1,로맨스,킹더랜드,15,1368579.0,7633.0,295.0
2,불륜복수,부부의 세계,21,1577691.0,7814.0,447.0
3,불륜복수,펜트하우스,32,3180746.0,31826.0,1394.0
4,재벌,상속자들,23,5229788.0,30892.0,1433.0
5,재벌,재벌집 막내아들,44,2218647.0,9678.0,554.0


In [11]:
def load_top_videos(engine, n: int = 10, table: str = 'longform_dramas') -> pd.DataFrame:
    """조회수 상위 영상 목록을 반환한다."""
    try:
        return pd.read_sql(
            text(f'''
                SELECT genre, keyword, title, channel, view_count, url
                FROM `{table}`
                ORDER BY view_count DESC
                LIMIT {n}
            '''),
            con=engine
        )
    except Exception as e:
        print(f'[ERROR] TOP 영상 조회 실패: {e}')
        raise


top10 = load_top_videos(engine)
print('=== 조회수 TOP 10 ===')
display(top10)

=== 조회수 TOP 10 ===


,genre,keyword,title,channel,view_count,url
0,재벌,상속자들,잘 봐 엄마들 기싸움이다! 박신혜 가짜 엄마 돈자랑에 기 눌려 아들에게 바로 일러바...,SBS Catch,13704196,https://www.youtube.com/watch?v=mkWwwti29jY
1,불륜복수,펜트하우스,펜트하우스 TOP9 공연 모음이 도착했습니다🎁 하늘이 내린 천상의 목소리는 누규~?...,SBS Catch,13320980,https://www.youtube.com/watch?v=FLjSG2dsqvA
2,재벌,상속자들,"레전드 드라마 [상속자들] Ep.12, 13 '영도파 VS 김탄파' / 'The H...",SBS,13218596,https://www.youtube.com/watch?v=K_2EcUcXWQs
3,재벌,상속자들,"레전드 드라마 [상속자들] Ep.16, 17 사라진 은상이와 전설의 그 키스' / ...",SBS,11881681,https://www.youtube.com/watch?v=Sn4R7YKIlU8
4,불륜복수,펜트하우스,[펜트하우스2] '엄마가 미안해.. 배로나를 잃은 오윤희의 뜨거운 눈물' / 'Th...,SBS,11405054,https://www.youtube.com/watch?v=mo30t3OpqWE
5,재벌,재벌집 막내아들,"날 평생 개처럼 부리다 비참하게 죽인 재벌가문의 막내아들로 ""회귀.환생""했다..! ...",지무비 : G Movie,9733687,https://www.youtube.com/watch?v=1wkqBs_xngc
6,재벌,상속자들,"레전드 드라마 [상속자들] Ep.18, 19 '이젠 보지 말자' / 'The Hei...",SBS,9384498,https://www.youtube.com/watch?v=FtUAa4hzDws
7,재벌,상속자들,"레전드 드라마 [상속자들] Ep.20 '그럼에도 우린, 직진' / 'The Heir...",SBS,8291569,https://www.youtube.com/watch?v=FsS9x6KBiFg
8,불륜복수,펜트하우스,[#펜트하우스] 시즌1 복습 1~10화 핵심만 요약! 정주행 생각에 벌써 떨리는군....,SBS Catch,7804084,https://www.youtube.com/watch?v=TycIYgaDdC4
9,재벌,상속자들,승부욕 생기게 훅 들어오는 츤츤매력 이보나💛 은근히 찌푸리는 미간에 끼고 싶어(¬‿...,SBS Catch,7661177,https://www.youtube.com/watch?v=4q4IO9_lxrI
